# 5차시 (4) 멀티 에이전트 — 직접 코드로 만들기

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

> **2차시(온라인)** 에서 우리는 프롬프트·RAG·에이전트를 **완성된 AI 서비스로 체험**했습니다.
> **오늘(오프라인)** 은 그걸 *서비스로 쓰는 게 아니라*, **직접 코드로 만들어 화면으로 확인**합니다.

앞 노트북에서 만든 챗봇/RAG는 "질문 하나 → 답 하나" 구조였습니다.
이번에는 한 걸음 더 나아가, **의도에 따라 갈라지고(라우팅) · 도구를 스스로 쓰는(tool use)** 에이전트를 코드로 만듭니다.

- **Part 1. 의도 분류 라우팅** — 질문의 *의도* 를 파악해 알맞은 처리로 보내기
- **Part 2. 도구 사용(tool use)** — 모델이 스스로 계산·검색 도구를 호출해 답하기
- **Part 3. 멀티 에이전트 & LangGraph** — 라우팅+도구를 엮는 큰 그림(개념)

> **에이전트(agent)** = 스스로 *계획하고, 도구를 쓰고, 반복* 하며 일을 해내는 AI. 오늘 그 핵심 두 조각(라우팅·도구 사용)을 손으로 만들어 봅니다.

## 0. 환경 세팅
라이브러리 설치 + API 키 입력 (앞 노트북과 동일).

> ⚠️ 이 노트북의 코드 셀은 대부분 **OpenAI API** 를 호출합니다. 반드시 **Colab 등 API 키가 설정된 환경에서 실행**하세요. (문법 검증용 로컬 환경에서는 실행되지 않습니다.)

In [ ]:
!pip install -qU langchain-openai langchain-core
print("설치 완료!")

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키 설정 완료" if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다.")

---
# Part 1 · 의도 분류 라우팅 (Routing)

실제 챗봇 서비스는 들어온 질문을 곧장 LLM에 던지지 않고, **먼저 의도를 분류** 한 뒤 알맞은 곳으로 보냅니다.
예: "환불해줘" → 환불 담당 / "오늘 날씨" → 검색 도구 / "우리 회사 휴가 규정" → RAG.

이것을 **라우팅(routing)** 이라 하고, 의도를 가르는 부분이 **라우터(router)** 입니다.
가장 간단한 방법은 **LLM에게 분류를 시키는 것** 입니다.

이제 그 라우터를 **직접 코드로 만들어 봅시다.** 먼저, 의도를 판별하는 함수 `classify_intent` 부터 손으로 작성합니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 과제 1 · 의도 분류 함수 <code>classify_intent</code> 직접 작성 (가이드형)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 덱에서 본 <b>'의도 분류'</b> 를 직접 구현합니다. 아래 함수 뼈대의 <code>TODO</code> 세 줄을 지우고, 여러분이 직접 본체를 채워 <code>classify_intent(message)</code> 를 완성하세요. (이 노트북의 <b>첫 과제</b>라 뼈대를 드립니다. 다음 과제부터는 빈 셀에서 처음부터 작성합니다.)<br><b>힌트</b> &nbsp; ① <code>INTENTS</code> 목록을 프롬프트에 넣어 "이 중 하나로만, 의도 단어만 출력" 하도록 지시하는 <b>프롬프트 문자열</b>을 만드세요(f-string). ② <code>router_llm</code> 을 호출(<code>invoke</code>)해 응답 텍스트를 받고 양끝 공백을 정리하세요. ③ 응답 안에 <code>INTENTS</code> 의 의도가 들어 있으면 그 의도를, 없으면 <code>"일반대화"</code> 를 <b>반환</b>하세요.<br><b>예상</b> &nbsp; 테스트 세 문장이 각각 <code>날씨</code> · <code>계산</code> · <code>일반대화</code> 로 분류돼 출력되면 성공입니다.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
from langchain_openai import ChatOpenAI

# 셋업: 분류 전용 LLM(창의성 0)과 의도 목록은 준비돼 있습니다.
router_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
INTENTS = ["날씨", "계산", "일반대화"]

def classify_intent(message):
    # 프롬프트는 드립니다: 의도 목록으로만 분류하고 "의도 단어만 출력" 하라는 지시 (f-string, 그대로 사용)
    prompt = f"""다음 사용자의 말을 아래 의도 중 하나로만 분류해줘. 다른 말 없이 의도 단어만 출력해.

가능한 의도: {', '.join(INTENTS)}

사용자: {message}
의도:"""
    result = ____          # ← router_llm 을 호출해 응답 텍스트를 받고 양끝 공백을 정리
    for it in INTENTS:
        if ____:           # ← 응답 안에 의도 it 가 들어 있으면
            return ____    # ← 그 의도를 반환
    return ____            # ← 아무 의도도 못 찾았을 때의 기본값

# 테스트 (완성하면 세 문장이 각각 날씨/계산/일반대화 로 갈립니다)
for msg in ["내일 서울 날씨 어때?", "123 곱하기 47은?", "심심한데 농담 하나 해줘"]:
    print(f"'{msg}'  ->  의도: {classify_intent(msg)}")

분류만으로는 끝이 아닙니다. 판별한 의도에 따라 **알맞은 처리기(handler)로 보내야** 라우팅이 완성됩니다.
아래 세 처리기는 미리 준비했습니다(날씨·계산은 예시 응답, 일반대화는 실제 LLM 답변).
이 처리기들을 엮는 **라우터 `route` 는 여러분이 직접 만듭니다.**

In [ ]:
chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 아래 3개 처리기는 준비돼 있습니다. (실제 서비스라면 각각 날씨 API·계산 도구·LLM 을 부릅니다)
def handle_weather(message):
    return "[날씨 처리기] 날씨 정보는 기상청 API에 연결하면 됩니다. (예: 내일 서울 맑음, 최고 28도)"

def handle_calc(message):
    return "[계산 처리기] 계산이 필요하군요. Part 2에서 '도구 사용'으로 실제 계산을 해봅니다."

def handle_chat(message):
    return "[일반대화] " + chat_llm.invoke(message).content

print("처리기 3개 준비 완료: handle_weather, handle_calc, handle_chat")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 과제 2 · 라우터 <code>route(message)</code> 직접 작성</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 덱에서 본 <b>'라우팅'</b> 을 직접 구현합니다. 위 <code>classify_intent</code> 로 의도를 얻고, 의도에 따라 알맞은 처리기를 호출해 그 결과를 돌려주는 <code>route(message)</code> 함수를 <b>빈 셀에서 처음부터</b> 작성하세요.<br><b>힌트</b> &nbsp; 먼저 <code>classify_intent(message)</code> 로 의도를 구하세요. 그다음 <b>if / elif / else</b> 로 분기해 — 의도가 날씨면 날씨 처리기, 계산이면 계산 처리기, 그 밖이면 일반대화 처리기를 호출하고 그 반환값을 <code>route</code> 가 그대로 <b>반환</b>하게 하세요.<br><b>예상</b> &nbsp; <code>route("오늘 부산 날씨 알려줘")</code> → <code>[날씨 처리기]…</code>, <code>route("안녕! 오늘 기분 어때?")</code> → <code>[일반대화]…</code> 처럼 의도별로 다른 처리기가 답하면 성공.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
# 과제 2: route(message) 를 완성하세요. 뼈대(if/elif/else)는 드리고, ____ 를 채웁니다.

def route(message):
    intent = ____             # ← classify_intent 로 의도 판별
    if ____:                  # ← 의도가 '날씨' 이면
        return ____           #    날씨 처리기(handle_weather) 결과를 반환
    elif ____:                # ← 의도가 '계산' 이면
        return ____           #    계산 처리기(handle_calc)
    else:
        return ____           # ← 그 밖이면 일반대화 처리기(handle_chat)

# --- 작성한 뒤 아래로 테스트 ---
print(route("오늘 부산 날씨 알려줘"))
print(route("안녕! 오늘 기분 어때?"))

> **연결 포인트**: 직전 노트북(BERT 분류)에서 *의도 데이터* 로 분류기를 학습했다면, 위 `classify_intent` 를 그 **BERT 분류기로 바꿔** 더 빠르고 저렴하게 라우팅할 수 있습니다(LLM 호출 비용↓).
> 또는 2~4차시의 **임베딩·유사도** 를 써서, 각 의도 예시 문장과의 유사도로 분류하는 **시맨틱 라우팅** 도 가능합니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 과제 3 · 라우터에 <code>'번역'</code> 의도 확장하기 (직접 작성)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 라우터는 필요할 때 의도를 <b>늘려</b> 갑니다. "영어로 번역해줘" 같은 말이 <b>번역 처리기</b>로 가도록, 새 의도 <code>'번역'</code> 을 라우터에 직접 추가하세요. 빈칸 채우기가 아니라 <b>필요한 함수들을 직접 작성</b>합니다.<br><b>힌트</b> &nbsp; 세 가지를 직접 만드세요. ① 의도 목록에 <code>'번역'</code> 을 넣은 새 분류 함수(예: <code>classify_intent_v2</code>) — 과제 1을 참고해 <code>INTENTS_V2</code> 로. ② 번역 처리기 <code>handle_translate(message)</code> — <code>chat_llm</code> 에게 "설명 없이 자연스러운 영어로 번역만" 하도록 지시하는 프롬프트로. ③ <code>'번역'</code> 분기를 포함한 새 라우터(예: <code>route_v2</code>).<br><b>예상</b> &nbsp; <code>route_v2("이 문장 영어로 번역해줘: 나는 학생입니다")</code> → <code>[번역] I am a student.</code> 처럼 번역/날씨/계산/잡담이 각각 알맞게 갈립니다.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
# 과제 3: '번역' 의도를 라우터에 직접 추가하세요. 아래 세 조각의 ____ 를 채웁니다.

INTENTS_V2 = ["날씨", "계산", "번역", "일반대화"]

# (1) '번역' 이 포함된 새 분류 함수 (과제 1을 참고하되 목록만 INTENTS_V2 로)
def classify_intent_v2(message):
    prompt = ____             # ← INTENTS_V2 를 넣어 "의도 단어만 출력" 지시하는 프롬프트
    result = ____             # ← router_llm 호출 후 응답 정리
    for it in INTENTS_V2:
        if it in result:
            return it
    return "일반대화"

# (2) 번역 처리기 (chat_llm 에게 "설명 없이 자연스러운 영어로 번역만" 지시)
def handle_translate(message):
    prompt = ____             # ← message 를 영어로 번역만 하라는 프롬프트
    return ____               # ← chat_llm 으로 번역 결과를 받아 앞에 "[번역] " 을 붙여 반환

# (3) '번역' 분기를 포함한 새 라우터
def route_v2(message):
    intent = ____             # ← classify_intent_v2 로 의도 판별
    if ____:                  # ← 번역이면
        return ____           #    번역 처리기
    elif intent == "날씨":
        return handle_weather(message)
    elif intent == "계산":
        return handle_calc(message)
    else:
        return handle_chat(message)

# --- 작성한 뒤 아래로 테스트 ---
for msg in ["이 문장 영어로 번역해줘: 나는 학생입니다", "내일 서울 날씨 어때?", "3 더하기 4는?", "안녕! 반가워"]:
    print("->", route_v2(msg))

---
# Part 2 · 도구 사용 (Tool Use / 함수 호출)

라우터로 의도를 *우리가* 나눠 줄 수도 있지만, 똑똑한 모델은 **스스로 필요한 도구를 골라** 쓰기도 합니다.
이를 **도구 사용(tool use)** 또는 **함수 호출(function calling)** 이라 합니다.

흐름:
1. 우리가 모델에게 "이런 도구들이 있어"라고 **도구 목록(함수)** 을 알려줍니다.
2. 질문이 들어오면 모델이 **"이 도구를 이런 인자로 써!"** 라고 알려줍니다.
3. 우리가 그 도구(함수)를 실제로 **실행** 하고, 결과를 모델에게 돌려줍니다.
4. 모델이 그 결과로 **최종 답** 을 만듭니다.

먼저 모델이 쓸 **도구(함수)** 두 개를 준비합니다: 계산기와 (가짜)검색. 이 부분은 예시로 드리고,
**여러분은 뒤에서 새 도구를 직접 만들어 붙입니다.**

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """수식을 계산한다. 예: '12 * (3 + 4)'. 사칙연산과 괄호를 지원한다."""
    try:
        # 안전을 위해 숫자와 기본 연산자만 허용
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "허용되지 않은 문자가 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def web_search(query: str) -> str:
    """인터넷에서 정보를 검색한다. 최신 정보나 사실 확인이 필요할 때 사용한다."""
    # 실제로는 검색 API(예: Tavily, SerpAPI)를 호출합니다.
    # 이번 실습에서는 키 없이 동작하도록 '가짜 검색 결과'를 돌려줍니다.
    fake_db = {
        "한경국립대": "한경국립대학교는 경기도 안성/평택에 캠퍼스를 둔 국립대학교입니다.",
        "파이썬": "파이썬(Python)은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어입니다.",
    }
    for key, val in fake_db.items():
        if key in query:
            return val
    return "(가짜 검색) 관련 결과를 찾지 못했습니다. 실제 서비스에서는 검색 API를 연결하세요."

print("도구 2개 준비 완료: calculator, web_search")

이제 모델에 도구를 **연결(bind)** 하고, 모델이 도구를 호출하면 실제로 실행해 결과를 돌려주는 **루프** 를 만듭니다.
이 루프가 바로 '에이전트' 의 핵심 동작입니다: *모델이 도구를 부르면 → 실행 → 결과를 다시 모델에게 → 최종 답*.
(이 실행 루프 `run_agent` 는 예시로 드립니다. 여러분의 과제는 **여기에 새 도구를 직접 만들어 붙이는 것**입니다.)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

tools = [calculator, web_search]
tools_by_name = {t.name: t for t in tools}

# 모델에게 '이런 도구를 쓸 수 있다'고 알려줌
agent_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def run_agent(question):
    messages = [HumanMessage(content=question)]

    # 1) 모델에게 질문 -> 모델이 '도구를 쓰자'고 할 수 있음
    ai_msg = agent_llm.invoke(messages)
    messages.append(ai_msg)

    # 2) 모델이 요청한 도구가 있으면 실제로 실행
    if ai_msg.tool_calls:
        for call in ai_msg.tool_calls:
            tool = tools_by_name[call["name"]]
            result = tool.invoke(call["args"])
            print(f"  [도구 실행] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        # 3) 도구 결과를 모델에게 다시 줘서 최종 답 생성
        final = agent_llm.invoke(messages)
        return final.content
    else:
        # 도구가 필요 없으면 바로 답
        return ai_msg.content

print("질문: 137 곱하기 256은 얼마야?")
print("답:", run_agent("137 곱하기 256은 얼마야?"))

In [ ]:
# 검색 도구가 필요한 질문
print("질문: 한경국립대가 어떤 학교야?")
print("답:", run_agent("한경국립대가 어떤 학교야?"))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">👀 관찰 · 모델이 어떤 도구를 고르는지 확인</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 이번 칸은 <b>작성 과제가 아니라 관찰</b>입니다. 아래 세 질문을 실행해, 모델이 <code>calculator</code> · <code>web_search</code> 중 무엇을 고르는지(또는 도구 없이 답하는지) <code>[도구 실행]</code> 로그로 확인하세요. 바로 다음 과제에서 여러분의 도구를 붙일 발판입니다.<br><b>힌트</b> &nbsp; 계산이 필요한 질문 → <code>calculator</code>, 사실·검색이 필요한 질문 → <code>web_search</code>, 감정·잡담 → 도구 없이 답. 모델은 각 도구의 <b>설명(docstring)</b> 을 읽고 고릅니다.<br><b>관찰 포인트</b> &nbsp; 첫 질문엔 <code>calculator</code>, 둘째엔 <code>web_search</code> 로그가 찍히고, 셋째엔 로그 없이 대화로 답하면 성공.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
# 세 질문을 각각 실행해 도구 선택을 관찰하세요.
test_questions = [
    "123 곱하기 45는 얼마야?",     # -> 계산기(calculator) 도구를 골라야 함
    "한경국립대는 어떤 학교야?",    # -> 검색(web_search) 도구를 골라야 함
    "오늘 기분이 좀 우울해",        # -> 도구 없이 그냥 대화로 답
]
for q in test_questions:
    print("질문:", q)
    print("답:", run_agent(q))
    print("-" * 45)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 과제 4 · 나만의 도구를 처음부터 만들어 에이전트에 붙이기 (직접 작성)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 덱에서 본 <b>'도구 사용'</b> 을 확장합니다. 모델이 스스로 고를 <b>새 도구</b>를 <code>@tool</code> 로 직접 정의하고, 아래 실행부의 <code>tools_v2</code> 에 추가해, 그 도구가 필요한 질문에서 <code>[도구 실행]</code> 로그가 찍히는지 확인하세요.<br><b>힌트</b> &nbsp; <code>@tool</code> 데코레이터를 붙인 함수를 만들되, <b>docstring(설명)</b> 에 '이 도구를 언제 쓰는지' 를 한 문장으로 분명히 쓰세요(모델은 이 설명을 읽고 고릅니다). 동작은 파이썬으로 간단히 — 아이디어: 글자 수 세기(<code>len</code>) · 문자열 뒤집기(<code>[::-1]</code>) · 섭씨↔화씨 변환 등. 만든 도구를 <code>tools_v2</code> 리스트에 넣고, 마지막 줄에 그 도구가 필요한 질문을 넣으세요.<br><b>예상</b> &nbsp; 예를 들어 '글자 수 세기' 도구라면 "'안녕하세요'는 몇 글자야?" 질문에 <code>[도구 실행]</code> 로그가 찍히고 답이 나옵니다.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
from langchain_core.tools import tool

# 과제 4: 새 도구를 직접 정의하세요. (@tool + docstring + 파이썬 동작 + 문자열 반환)
@tool
def ____(____: str) -> str:       # ← 도구 이름과 입력 인자 이름을 정하세요
    """____"""                    # ← 이 도구를 '언제 쓰는지' 한 문장 (모델이 읽고 고름)
    ____                          # ← 파이썬으로 동작을 구현하고 결과를 문자열로 반환

# --- 아래는 준비된 실행부입니다: tools_v2 에 여러분이 만든 도구를 추가하세요 ---
tools_v2 = [calculator, web_search, ____]   # ← ____ 자리에 위에서 만든 도구 이름
tools_by_name.update({t.name: t for t in tools_v2})
agent_llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools_v2)

def run_agent2(question):
    messages = [HumanMessage(content=question)]
    ai_msg = agent_llm2.invoke(messages); messages.append(ai_msg)
    if ai_msg.tool_calls:
        for call in ai_msg.tool_calls:
            result = tools_by_name[call["name"]].invoke(call["args"])
            print(f"  [도구 실행] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        return agent_llm2.invoke(messages).content
    return ai_msg.content

# 여러분이 만든 도구가 필요한 질문을 직접 던져 보세요.
print(run_agent2(____))   # ← ____ 자리에 그 도구가 필요한 질문을 문자열로

> **실제 인터넷 검색을 붙이려면**: 위 `web_search` 의 가짜 부분을 실제 검색 API로 바꾸면 됩니다.
> 대표적으로 **Tavily**(`pip install langchain-tavily`, 무료 키 발급)나 SerpAPI 등을 연결합니다. 키가 필요하므로 이번 수업에서는 가짜 검색으로 흐름만 익혔습니다.

---
# Part 3 · (개념) 멀티 에이전트와 LangGraph

지금까지 **직접 만든** 라우팅과 도구 사용을 여러 개 엮으면 **멀티 에이전트 시스템** 이 됩니다.
예: *질문 의도 분류 → (검색 에이전트 / RAG 에이전트 / 계산 에이전트) → 결과를 정리하는 에이전트*.

이런 흐름을 **그래프(graph)** 로 그려 관리하는 대표 도구가 **LangGraph** 입니다.
- **노드(node)**: 하나의 작업(에이전트/함수). 예: '의도 분류', '검색', '답 생성'.
- **엣지(edge)**: 다음에 어디로 갈지(조건 분기 포함).
- **상태(state)**: 노드들이 공유하는 정보(대화·중간 결과).

오늘 여러분이 손으로 만든 `route()` 함수가 바로 작은 LangGraph 한 장입니다.

```
           ┌──────────────┐
  질문 ──▶  │  의도 분류    │
           └──────┬───────┘
        ┌─────────┼─────────┐
        ▼         ▼         ▼
   [날씨 검색] [계산 도구] [RAG/대화]
        └─────────┼─────────┘
                  ▼
             최종 답 정리
```

**언제 무엇을 쓰나(2026 기준, 개념만)**
- 복잡하고 분기·상태가 많은 워크플로 → **LangGraph**
- 빠른 프로토타입, 역할 기반 팀 구성 → **CrewAI**
- RAG 중심 검색 에이전트 → **LlamaIndex**

> 초급 단계에서는 "라우팅 + 도구 사용" 개념만 확실히 잡으면 충분합니다. 프레임워크는 필요해질 때 골라 쓰면 됩니다.

---
## 🔑 막히면 펼쳐보기 (힌트)

과제가 막힐 때만 펼쳐 참고하세요. **완성 코드가 아니라 방향 힌트**입니다. 먼저 스스로 고민해 보고 열어 주세요.

<details>
<summary>과제 1 · <code>classify_intent</code> 가 막히면</summary>

- LLM 호출은 <code>router_llm.invoke(프롬프트)</code>. 그 결과에서 <code>.content</code> 를 꺼내고 <code>.strip()</code> 으로 양끝 공백을 정리합니다.
- 의도 찾기: <code>for it in INTENTS:</code> 를 돌며 <code>it in result</code> (문자열 포함) 이 참이면 <code>return it</code>.
- 반복이 끝나도록 못 찾으면 마지막 줄에서 기본값 <code>"일반대화"</code> 를 반환합니다.
</details>

<details>
<summary>과제 2 · <code>route</code> 가 막히면</summary>

- 먼저 <code>classify_intent(message)</code> 로 의도를 변수에 담습니다.
- 조건은 <code>intent == "날씨"</code> · <code>intent == "계산"</code> 처럼 비교합니다.
- 각 가지에서 <code>handle_weather</code> · <code>handle_calc</code> · <code>handle_chat</code> 중 하나를 <b>호출한 값</b>을 <code>return</code> 하세요(함수 자체가 아니라 호출 결과).
</details>

<details>
<summary>과제 3 · '번역' 확장이 막히면</summary>

- 분류 함수의 프롬프트는 과제 1과 같은 모양이되 목록만 <code>INTENTS_V2</code> 로 바꾸고, LLM 호출도 과제 1과 동일합니다.
- 번역 처리기의 프롬프트 예: <code>f"다음 문장을 설명 없이 자연스러운 영어로 번역만 해줘:\n{message}"</code>. 반환은 <code>chat_llm.invoke(...).content</code> 앞에 <code>"[번역] "</code> 을 붙입니다.
- 새 라우터는 과제 2의 if/elif 사슬 맨 앞에 <code>if intent == "번역":</code> 가지를 하나 더 답니다.
</details>

<details>
<summary>과제 4 · 새 <code>@tool</code> 이 막히면</summary>

- 형태: <code>@tool</code> 바로 아래 <code>def 이름(text: str) -> str:</code>, 그 첫 줄에 <b>docstring</b>("언제 쓰는 도구인지" 한 문장 — 모델이 이 설명을 읽고 고릅니다).
- 동작 아이디어: 글자 수 <code>len(text)</code> · 뒤집기 <code>text[::-1]</code> · 섭씨→화씨 <code>float(text)*9/5+32</code>. 마지막에 결과를 <b>문자열</b>로 <code>return</code>.
- 실행부의 <code>tools_v2</code> 리스트에 만든 함수 이름을 넣으면 <code>agent_llm2</code> 가 그 도구를 쓸 수 있습니다. 마지막 줄 질문은 그 도구가 꼭 필요한 문장으로 던지세요.
</details>

---
## 마무리

오늘 (4)번 노트북에서 **직접 코드로 작성한 것**:
- **`classify_intent`**: 프롬프트로 LLM에게 의도를 분류시키는 함수 (과제 1).
- **`route`**: 의도에 따라 알맞은 처리기로 분기하는 라우터 (과제 2), 그리고 `'번역'` 의도로 확장 (과제 3).
- **나만의 `@tool`**: 모델이 스스로 고르는 새 도구를 만들어 에이전트에 연결 (과제 4).

> **2차시에서 서비스로 봤던 것**(챗봇·RAG·에이전트·NotebookLM)을, 오늘은 **부품 단위로 직접 코드로** 만들어 봤습니다. 라우터도 도구도 결국 우리가 짠 함수입니다.

**5차시 전체 정리**: 프롬프트 엔지니어링 → 나만의 챗봇 → RAG 챗봇 → BERT 분류 → 멀티 에이전트.
작은 부품(프롬프트·임베딩·검색·분류·도구)을 조합하면 점점 똑똑한 AI 서비스가 됩니다. 수고하셨습니다!